# Proteome exploration with ESMC embeddings

A **single** scanpy graph drives everything: a KNN graph on the ESMC embeddings, one UMAP layout, and Leiden clusters — same neighbors graph, fixed seed. SAE features are tested for enrichment per cluster (proteins as "cells", SAE features as "genes") and richly annotated from the ESM Atlas.

**Prerequisites:** run the embedding step first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

> **Another species?** This notebook is config-driven — set `OCH_ANNOTATE_CONFIG=config/<species>.yaml` before launching (defaults to octopus). All column references come from the config's `baserow` roles (`id_column`, `name_column`, `ortholog_column`, `orthogroup_column`); no cell edits needed.

In [ ]:
import os as _os
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config(_os.environ.get("OCH_ANNOTATE_CONFIG", "../config/octopus_chierchiae.yaml"))
df = load_embeddings(cfg, prefer_cache=True)   # backfills Baserow metadata cols not in cache
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")

# --- Column roles from config (species-agnostic; a new species needs no edits here,
#     only its own config + `OCH_ANNOTATE_CONFIG=config/<species>.yaml`) ---
B = cfg.baserow
ID_COL, NAME_COL, ORTHO_COL, OG_COL = B.id_column, B.name_column, B.ortholog_column, B.orthogroup_column
def _cols(cands):
    """Named columns that actually exist in df, in order, skipping None/dupes."""
    out, seen = [], set()
    for c in cands:
        if c and c in df.columns and c not in seen:
            seen.add(c); out.append(c)
    return out
SEARCH_COLS = _cols([ID_COL, NAME_COL, ORTHO_COL, OG_COL, "gene_id"])
HOVER_COLS  = _cols([ID_COL, "gene_id", NAME_COL, ORTHO_COL, OG_COL, "chromosome"])
OG_LABELS   = {OG_COL: "Orthogroup"} if OG_COL else {}
_sp = cfg.sae.feature_store_path or "data/sae_feature_matrix.npz"
STORE_PATH  = _sp if _os.path.isabs(_sp) else _os.path.join("..", _sp)
df.head()

## One graph: KNN → UMAP → Leiden

Defaults match the original UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine). Tune **granularity**: `N_NEIGHBORS`/`MIN_DIST` shape the UMAP, `LEIDEN_RES` the cluster count.

In [ ]:
import scanpy as sc
from och_annotate.analysis import build_anndata, sae_enrichment, plot_umap

SEED        = 0
N_NEIGHBORS = 15
MIN_DIST    = 0.1
METRIC      = "cosine"
LEIDEN_RES  = 1.0       # clustering granularity (higher = more clusters)

adata = build_anndata(df)
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
from och_annotate.analysis import plot_umap_searchable

# UMAP colored by chromosome, with a client-side gene search box (works in the
# exported HTML): searches gene/ortholog/orthogroup ids; matches are ringed.
# Fixed square, centered plot. embed_js=True loads plotly.js once for the document.
plot_umap_searchable(coords, color=("chromosome" if "chromosome" in coords.columns else "leiden"),
          hover=HOVER_COLS + ["leiden"],
          search_fields=SEARCH_COLS,
          labels=OG_LABELS,
          title=f"{cfg.name} — UMAP (chromosome)", embed_js=True)

In [ ]:
# Same UMAP colored by Leiden cluster, same searchable box. embed_js=False:
# reuse the plotly.js already loaded above (avoids embedding it twice).
plot_umap_searchable(coords, color="leiden",
          hover=HOVER_COLS + ["leiden"],
          search_fields=SEARCH_COLS,
          labels=OG_LABELS,
          title=f"{cfg.name} — UMAP (Leiden clusters)", embed_js=False)

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix, annotated from the **ESM Atlas**: `label`, `category`, `activation_pattern`, `exemplar_protein_families`, top **SwissProt** proteins, and `uniref90_idf` — used to **IDF-weight** markers toward specific (rare) features.

In [ ]:
import pandas as pd
from och_annotate.atlas import fetch_feature_descriptions

enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Rich per-feature Atlas metadata for the enriched features (concurrent, cached; no credits)
feat_ids = sorted(enrich["sae_feature"].astype(int).unique())
meta = fetch_feature_descriptions(feat_ids, cache_path="../data/sae_feature_metadata.parquet")
keep = ["feature", "label", "category", "activation_pattern",
        "exemplar_protein_families", "uniref90_idf", "swissprot_top"]
meta = meta[keep].copy(); meta["feature"] = meta["feature"].astype(str)

enrich["feature"] = enrich["sae_feature"].astype(str)
enrich = enrich.merge(meta, on="feature", how="left").drop(columns="feature")

# IDF-weighting: upweight features that are rare across UniRef90 (more specific).
enrich["idf"] = pd.to_numeric(enrich["uniref90_idf"], errors="coerce").fillna(1.0)
enrich["score_idf"] = enrich["scores"] * enrich["idf"]

enrich.to_csv("../data/cluster_sae_enrichment.csv", index=False)
print(f"Annotated {len(feat_ids)} features (category / IDF / exemplars / SwissProt); "
      f"{len(enrich)} rows across {enrich['leiden'].nunique()} clusters")

In [ ]:
from och_annotate.analysis import table_searchable

# Per-cluster functional profile: the category mix of each cluster's top-15 features
profile = (enrich.assign(cluster=enrich["leiden"].astype(int))
                 .groupby("cluster")["category"]
                 .apply(lambda s: ", ".join(f"{c} ({n})" for c, n in
                        s.fillna("(uncat)").replace("", "(uncat)").value_counts().head(4).items()))
                 .rename("top_feature_categories").reset_index())
table_searchable(
    profile, title="Per-cluster functional profile — dominant feature categories",
    caption="Category mix of each cluster's top-15 Wilcoxon markers. "
            "Searchable / paginated (DataTables via CDN; data embedded).",
    page_size=15, max_colwidth=90, order=[["cluster", "asc"]])

In [ ]:
# Top-5 per cluster, ranked by the IDF-WEIGHTED score (specific features rise).
top5 = (enrich.assign(cluster=enrich["leiden"].astype(int))
              .sort_values(["cluster", "score_idf"], ascending=[True, False])
              .groupby("cluster", observed=True).head(5).copy())
top5["rank"] = top5.groupby("cluster").cumcount() + 1
view = top5[["cluster", "rank", "sae_feature", "label", "category", "scores", "idf", "score_idf"]]

table_searchable(
    view, title="Cluster SAE-feature enrichment — top 5 per cluster (IDF-weighted)",
    caption="Wilcoxon markers re-ranked by score_idf = score × uniref90_idf. "
            "Searchable / sortable / paginated (DataTables via CDN; data embedded).",
    page_size=15,
    formats={"scores": "{:.1f}", "idf": "{:.2f}", "score_idf": "{:.1f}"},
    max_colwidth=70, order=[["cluster", "asc"], ["score_idf", "desc"]])

In [ ]:
# Rich context for each cluster's lead (IDF-weighted top) feature
lead = top5[top5["rank"] == 1].sort_values("cluster")
for r in lead.itertuples():
    ap = (str(r.activation_pattern) or "").strip().replace("\n", " ")
    ex = (str(r.exemplar_protein_families) or "").strip().splitlines()
    print(f"\u2501\u2501 cluster {r.cluster}  [{r.sae_feature}] {r.label}  ({r.category})")
    print(f"     activation : {ap[:220]}")
    print(f"     exemplars  : {(ex[0][:200] if ex else '')}")
    print(f"     SwissProt  : {r.swissprot_top}")

In [ ]:
# Dotplot of marker SAE features across clusters (Wilcoxon ranking)
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

## Per-cluster feature profile — salience × ubiquity

A second lens on cluster identity, ranked by **mean normalized activation** (what ESMC finds most *salient* across the cluster) with **occurrence** = the fraction of members in which the feature is active. Universal features (occurrence ≈ 100%) define the family; partial ones (≈ 30–60%) flag subgroups or domain variants. This complements the differential Wilcoxon markers above. Reporting the **top 10** per cluster — ranks 6–15 are where subfamily discrimination lives.

> **Residue regions** (start/end/peak per feature) would be the next upgrade, but our cache max-pools SAE activations to one value per protein — positions were discarded. Recovering them needs a per-residue SAE re-run (Biohub credits).

In [ ]:
from och_annotate.analysis import cluster_feature_profile

# Top-10 features per cluster by mean normalized activation (+ occurrence rate)
profile = cluster_feature_profile(adata, groupby="leiden", n=10)

# Atlas labels/category for the profile features (cached; no Biohub credits)
pf_ids = sorted(profile["sae_feature"].unique())
pmeta = fetch_feature_descriptions(pf_ids, cache_path="../data/sae_feature_metadata.parquet")
plabel = dict(zip(pmeta["feature"].astype(int), pmeta["label"]))
pcat = dict(zip(pmeta["feature"].astype(int), pmeta["category"]))
profile["label"] = profile["sae_feature"].map(plabel)
profile["category"] = profile["sae_feature"].map(pcat)
profile.to_csv("../data/cluster_feature_profile.csv", index=False)
print(f"Profiled {profile['cluster'].nunique()} clusters x top-10 features "
      f"({len(pf_ids)} unique features)")

In [ ]:
# Grouped top-10 profile per cluster: salience (mean_activation) + ubiquity (occurrence)
pv = (profile.assign(cluster=lambda d: d["cluster"].astype(int))
             .sort_values(["cluster", "rank"])
             [["cluster", "rank", "sae_feature", "label", "category",
               "mean_activation", "occurrence"]])

table_searchable(
    pv, title="Per-cluster feature profile — top 10 by salience × ubiquity",
    caption="mean_activation = salience across cluster members; occurrence = fraction active. "
            "Searchable / sortable / paginated (DataTables via CDN; data embedded).",
    page_size=15,
    formats={"mean_activation": "{:.3f}", "occurrence": "{:.0%}"},
    max_colwidth=70, order=[["cluster", "asc"], ["rank", "asc"]])

## Per-candidate feature report

The per-protein workflow: **top-10 normalized features** with Atlas labels, plus **residue regions** (start–end, peak) for the architecture-bearing top few. The `residues` column is wired but blank until a per-residue SAE run populates it (`sae.residue_regions: true` — same Biohub call, no extra cost; needs a re-run).

In [ ]:
from och_annotate.analysis import candidate_feature_report
from och_annotate.atlas import fetch_all_features

fd = fetch_all_features(cache_path="../data/sae_feature_dictionary.parquet")
full_labels = dict(zip(fd["feature"].astype(int), fd["label"]))

cand = df.iloc[0]   # example candidate; swap in any row / transcript_id
rep = candidate_feature_report(cand["sae_top_features"], labels=full_labels, n=10)
print(f"Candidate {cand.get(ID_COL,'?')}  ({cand.get(NAME_COL,'') if NAME_COL else ''})")
display(rep)
print("residue regions:", "present" if rep["residues"].notna().any()
      else "pending a per-residue SAE run (set sae.residue_regions=true)")

### Notes on the Atlas annotations

All metadata comes from the public **ESM Atlas** feature API
(`biohub.ai/esm/protein/api/v1alpha1/features/{idx}`) via
`och_annotate.atlas.fetch_feature_descriptions` — cached under `data/`, **not** charged
against Biohub embedding credits. The grouped table is ranked by
`score_idf = wilcoxon_score × uniref90_idf` so cluster-specific (rare) features rise above
ubiquitous ones; the lead-feature block adds activation pattern, exemplar families and
reviewed-UniProt examples. Full table: `data/cluster_sae_enrichment.csv`.

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- GO-enrich each cluster from the SwissProt example proteins.
- Tune `LEIDEN_RES`, `N_NEIGHBORS`, `MIN_DIST`.